# Vehicle Routing Problem con Algoritmos Genéticos

Cuaderno listo para **Google Colab** o **JupyterLab**.

Incluye:
- representación permutacional de clientes,
- decodificación a rutas por capacidad,
- fitness con penalización por exceso de vehículos,
- cruce OX,
- mutación por intercambio,
- gráfico de convergencia,
- visualización de rutas y cargas.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import math


## Datos del problema


In [ ]:
deposito = (50, 50)

clientes = {
    1: {'coord': (20, 60), 'demanda': 4},
    2: {'coord': (18, 54), 'demanda': 6},
    3: {'coord': (30, 40), 'demanda': 5},
    4: {'coord': (60, 20), 'demanda': 7},
    5: {'coord': (65, 35), 'demanda': 3},
    6: {'coord': (80, 55), 'demanda': 4},
    7: {'coord': (72, 70), 'demanda': 6},
    8: {'coord': (45, 75), 'demanda': 5},
}

lista_clientes = list(clientes.keys())
capacidad_vehiculo = 15
max_vehiculos = 4

tam_poblacion = 100
n_generaciones = 150
prob_cruce = 0.85
prob_mutacion = 0.10
tam_torneo = 3

random.seed(42)
np.random.seed(42)


## Funciones auxiliares


In [ ]:
def distancia(p1, p2):
    return math.dist(p1, p2)

def crear_individuo():
    individuo = lista_clientes.copy()
    random.shuffle(individuo)
    return individuo

def decodificar(individuo):
    rutas = []
    ruta_actual = []
    carga_actual = 0

    for cliente in individuo:
        demanda = clientes[cliente]['demanda']
        if carga_actual + demanda <= capacidad_vehiculo:
            ruta_actual.append(cliente)
            carga_actual += demanda
        else:
            rutas.append(ruta_actual)
            ruta_actual = [cliente]
            carga_actual = demanda

    if ruta_actual:
        rutas.append(ruta_actual)

    return rutas

def distancia_ruta(ruta):
    if not ruta:
        return 0
    total = 0
    punto_actual = deposito
    for cliente in ruta:
        coord_cliente = clientes[cliente]['coord']
        total += distancia(punto_actual, coord_cliente)
        punto_actual = coord_cliente
    total += distancia(punto_actual, deposito)
    return total

def distancia_total(rutas):
    return sum(distancia_ruta(ruta) for ruta in rutas)

def fitness(individuo):
    rutas = decodificar(individuo)
    total = distancia_total(rutas)
    penalizacion = 0
    if len(rutas) > max_vehiculos:
        penalizacion += 1000 * (len(rutas) - max_vehiculos)
    return -(total + penalizacion)


## Operadores genéticos


In [ ]:
def seleccion_torneo(poblacion):
    candidatos = random.sample(poblacion, tam_torneo)
    return max(candidatos, key=fitness).copy()

def cruce_orden(padre1, padre2):
    if random.random() >= prob_cruce:
        return padre1.copy(), padre2.copy()
    n = len(padre1)
    a, b = sorted(random.sample(range(n), 2))
    hijo1 = [-1] * n
    hijo2 = [-1] * n
    hijo1[a:b] = padre1[a:b]
    hijo2[a:b] = padre2[a:b]

    def completar(hijo, otro_padre):
        pos = 0
        for gen in otro_padre:
            if gen not in hijo:
                while hijo[pos] != -1:
                    pos += 1
                hijo[pos] = gen
        return hijo

    hijo1 = completar(hijo1, padre2)
    hijo2 = completar(hijo2, padre1)
    return hijo1, hijo2

def mutar_intercambio(individuo):
    mutado = individuo.copy()
    if random.random() < prob_mutacion:
        i, j = random.sample(range(len(mutado)), 2)
        mutado[i], mutado[j] = mutado[j], mutado[i]
    return mutado


## Ejecución del algoritmo genético


In [ ]:
poblacion = [crear_individuo() for _ in range(tam_poblacion)]

mejores_costos = []
promedios_costos = []
mejor_individuo_global = None
mejor_fit_global = -float('inf')

for generacion in range(n_generaciones):
    nueva_poblacion = []
    while len(nueva_poblacion) < tam_poblacion:
        padre1 = seleccion_torneo(poblacion)
        padre2 = seleccion_torneo(poblacion)
        hijo1, hijo2 = cruce_orden(padre1, padre2)
        hijo1 = mutar_intercambio(hijo1)
        hijo2 = mutar_intercambio(hijo2)
        nueva_poblacion.append(hijo1)
        if len(nueva_poblacion) < tam_poblacion:
            nueva_poblacion.append(hijo2)
    poblacion = nueva_poblacion
    fitness_poblacion = [fitness(ind) for ind in poblacion]
    mejor_generacion = max(poblacion, key=fitness)
    mejor_fit = fitness(mejor_generacion)
    promedio_fit = np.mean(fitness_poblacion)
    mejores_costos.append(-mejor_fit)
    promedios_costos.append(-promedio_fit)
    if mejor_fit > mejor_fit_global:
        mejor_fit_global = mejor_fit
        mejor_individuo_global = mejor_generacion.copy()

rutas_finales = decodificar(mejor_individuo_global)
costo_final = distancia_total(rutas_finales)

print('Mejor cromosoma encontrado:', mejor_individuo_global)
print('Cantidad de rutas:', len(rutas_finales))
print('Distancia total:', round(costo_final, 2))

for i, ruta in enumerate(rutas_finales, start=1):
    carga = sum(clientes[c]['demanda'] for c in ruta)
    print(f'Ruta {i}: {ruta} | carga={carga} | distancia={round(distancia_ruta(ruta), 2)}')


## Curva de convergencia


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_costos, label='Mejor costo')
plt.plot(promedios_costos, label='Costo promedio')
plt.xlabel('Generación')
plt.ylabel('Distancia total')
plt.title('Evolución del algoritmo genético para VRP')
plt.legend()
plt.grid(True)
plt.show()


## Visualización de rutas finales


In [ ]:
plt.figure(figsize=(8, 8))
colores = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']

plt.scatter(deposito[0], deposito[1], s=300, marker='s')
plt.text(deposito[0], deposito[1], 'Depósito', ha='left', va='bottom')

for cliente, data in clientes.items():
    x, y = data['coord']
    plt.scatter(x, y, s=150)
    plt.text(x, y, f'C{cliente}', ha='right', va='bottom')

for idx, ruta in enumerate(rutas_finales):
    puntos = [deposito] + [clientes[c]['coord'] for c in ruta] + [deposito]
    xs = [p[0] for p in puntos]
    ys = [p[1] for p in puntos]
    plt.plot(xs, ys, linewidth=2, label=f'Ruta {idx+1}')

plt.xlabel('X')
plt.ylabel('Y')
plt.title('Rutas finales encontradas')
plt.legend()
plt.grid(True)
plt.show()


## Carga por ruta


In [ ]:
cargas = [sum(clientes[c]['demanda'] for c in ruta) for ruta in rutas_finales]
etiquetas = [f'Ruta {i+1}' for i in range(len(rutas_finales))]

plt.figure(figsize=(8, 5))
plt.bar(etiquetas, cargas)
plt.axhline(capacidad_vehiculo, linestyle='--', label='Capacidad')
plt.xlabel('Rutas')
plt.ylabel('Carga')
plt.title('Carga por ruta')
plt.legend()
plt.grid(axis='y')
plt.show()


## Ejercicio sugerido

Modificar:
- las demandas de clientes,
- la capacidad del vehículo,
- la cantidad máxima de vehículos,
- la cantidad de generaciones,

y analizar cómo cambian las rutas finales y la distancia total.
